In [6]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio

colors = [
    ROOT.kRed + 1,
    ROOT.kBlue + 1,
    ROOT.kGreen + 2,
    ROOT.kOrange + 7,
    ROOT.kMagenta + 1,
    ROOT.kCyan + 1,
    ROOT.kViolet + 2,
    ROOT.kPink + 7,
    ROOT.kTeal + 3,
    ROOT.kAzure + 2,
    ROOT.kSpring + 5,
]
siphras = ['A', 'B']

In [7]:
files = sorted((project_root/'data/260518').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]
#print(acqs)
Abackground = acqs[12]
Bbackground = acqs[25]
Acalibration = acqs[0]
Bcalibration = acqs[13]
backgrounds = [Abackground, Bbackground]
calibrations = [Acalibration, Bcalibration]


In [8]:
# Create histograms for background and each distance for both siphras. Create background subtracted histograms. 
hists = {}
#distances_cm = list(range(3, 14))
for i in range(2):
    siphra = siphras[i]
    bg = backgrounds[i]
    cal = calibrations[i]

    #Background
    hist_bg = ROOT.TH1F(f"{siphra}_Background", "", BITS12, 0 , BITS12)
    hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_bg.GetXaxis().SetTitle("ADC channel number")
    hist_bg.GetYaxis().SetTitle(r"Counts")
    hists[f"{siphra}_Background"] = hist_bg
    del hist_bg

    hist_cal = ROOT.TH1F(f"{siphra}_Calibration", "", BITS12, 0 , BITS12)
    hist_cal.Fill(cal['s']/len(cal.active_chs))
    hist_cal.GetXaxis().SetTitle("ADC channel number")
    hist_cal.GetYaxis().SetTitle(r"Counts")
    hists[f"{siphra}_Calibration"] = hist_cal
    del hist_cal

    

Warning in <TROOT::Append>: Replacing existing TH1: A_Calibration (Potential memory leak).


In [9]:
#Draw one histogram for each distance + siphra combination

if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()

Yinit = 0.82 # For stat boxes

cv1 = ROOT.TCanvas('cv1', 'cv1', 3600, 1200)
cv1.Divide(2,1)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(2)]

for i, siphra in enumerate(siphras):  
    lg = lgds[i]
    cv1.cd(i+1)
    cv1.cd(i+1).SetLogy()
    bg = hists[f"{siphra}_Background"] 
    
    lg.AddEntry(bg, "Background")
    
    bg.SetLineColor(colors[1])
    bg.SetTitle("Background Radiation")


    # Drw histograms
    bg.Draw("hist")
    

    ROOT.gPad.Update()
    #lg.Draw()
    cv1.Draw()
    


In [10]:
from analysis import *

#Calibration based on the 3 peaks of the Na-22 spectrum
src = "Na-22"
c_fits = []
for siphra in siphras:
    hist = hists[f"{siphra}_Calibration"]
    energy_ranges = [(150,250), (420, 520), (590, 690)]
    energies = NA22_MEV
    
    c_fit = calibration_fit(hist, energy_ranges, energies)
    print(c_fit)
    c_fits.append(c_fit)

[0.0015069519390730635, 1.4460225231200314]
[0.0023456966938449077, 0.40720361007117006]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1518.98
NDf                       =           97
Edm                       =  9.11089e-06
NCalls                    =          785
Constant                  =      17732.8   +/-   2653.23     
Mean                      =     -573.685   +/-   17.7429     
Sigma                     =      301.597   +/-   2.05283      	 (limited)
****************************************
         Invalid FitResult  (status = 4 )
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      405.558
NDf                       =           97
Edm                       =    0.0524392
NCalls                    =         1349
Constant                  =  4.38815e+06   +/-   974464      
Mean                      =     -198.604   +/-   19.2659     
Sigma                     =      141

Warning in <Fit>: Abnormal termination of minimization.
Warning in <Fit>: Abnormal termination of minimization.
